**Without Pydantic → Python doesn't automatically enforce type hints.**

In [1]:
# before pydantic 

class Student:
    def __init__(self, name: str, age: int, gpa: float):
        self.name = name 
        self.age = age 
        self.gpa = gpa 

s = Student(name = 99, age = "Twenty", gpa = False)
print(s.name, " ", s.age, " ", s.gpa)

99   Twenty   False


**With Pydantic → data is validated when the model is created.**

In [2]:
# after pydantic 

from pydantic import BaseModel 

class Student(BaseModel):
    name:str 
    age:int
    gpa:float

s1 = Student(name = "Ram", age=21, gpa=2.9)
print(s1)

name='Ram' age=21 gpa=2.9


In [ ]:
s2 =  Student(name = 99, age = "Twenty", gpa = "excellent")
print(s2)

`ValidationError`

Pydantic found **3 validation errors**:

- **name:** Expected string, received integer (`99`).
- **age:** Expected integer, received `"Twenty"`.
- **gpa:** Expected float, received `"excellent"`.

**Conclusion:** Pydantic raises `ValidationError` when input data has invalid types or values.


`Real world approach, in API data will arrives in Dictionary format. `


> **Validation & Coercion**

Pydantic commonly attempts to convert compatible input types to the expected type.

In [ ]:
raw_data = {
    "name":"Sham",
    "age":20,
    "gpa":3.5
}

s2 = Student(**raw_data)    #** unpacks a dictionary into keyword arguments.
print(s2)

name='Sham' age=20 gpa=3.5


In [5]:
raw_data1 = {
    "name":"Sham",
    "age":"21",
    "gpa":"21.5"
}

s3 = Student(**raw_data1)
print(s3)
print(type(s3.name), type(s3.age), type(s3.gpa))

name='Sham' age=21 gpa=21.5
<class 'str'> <class 'int'> <class 'float'>


In [ ]:
raw_data2 = {
    "name":"Sham",
    "age":21.5,
    "gpa":"21.5"
}

s3 = Student(**raw_data2)
print(s3)

`ValidationError`

Pydantic found **1 validation error**:

- **age:** Expected an integer, but received `21.5` (float).

**Reason:** Pydantic cannot convert a fractional number into an integer automatically.


> #### **ValidationError** 

In [7]:
from pydantic import ValidationError 

raw_data4 = {
    "name" : 1234, 
    "age":"Twenty"
}

try:
    s4 = Student(**raw_data4)
except ValidationError as e: 
    print(f"Total number of errors found: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of errors found: 3
Fields : ('name',)  , Type : string_type ,  Message: Input should be a valid string
Fields : ('age',)  , Type : int_parsing ,  Message: Input should be a valid integer, unable to parse string as an integer
Fields : ('gpa',)  , Type : missing ,  Message: Field required



**Important Pydantic Methods to Remember**

| Method | Purpose |
|---|---|
| `e.error_count()` | Returns the number of validation errors |
| `e.errors()` | Returns detailed errors as a list |
| `err['loc']` | Identifies the field causing the error |
| `err['type']` | Identifies the error category |
| `err['msg']` | Returns the error message |


In [20]:
"""
Try writing a small Product(BaseModel) in your notebook with fields: 
product_id: int, name: str, price: float, in_stock: bool, 
and test it with:

Valid coerced inputs (product_id="50", price="199.99", in_stock="1").
A broken input inside try / except ValidationError as e: and print e.error_count().
"""

from pydantic import BaseModel, ValidationError

class Product(BaseModel):
    product_id: int
    name: str
    price: float
    in_stock: bool
    
values = {
    "product_id":"50",
    "price":"199.99",
    "in_stock":"1"
}

try:
    p1 = Product(**values)
except ValidationError as e:
    print(f"Total number of errors found: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of errors found: 1
Fields : ('name',)  , Type : missing ,  Message: Field required


> **Case A - Required and Non-nullable**

what happens when value is not provided in dict 

In [ ]:
from pydantic import BaseModel, ValidationError 

class Product(BaseModel):
    product_id: int
    name: str 

value = {
    "product_id" : "50"
}

try:
    p1 = Product(**value)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of error: 1
Fields : ('name',)  , Type : missing ,  Message: Field required


what happens if value is None

In [16]:
class Product(BaseModel):
    product_id: int
    name: str

value = {
    "product_id" : "50",
    "name": None
}

try:
    p1 = Product(**value)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of error: 1
Fields : ('name',)  , Type : string_type ,  Message: Input should be a valid string


> **Case B - Required and nullable** 

what happens when value is not provided in dict 

In [13]:
from pydantic import BaseModel, ValidationError 

class Product(BaseModel):
    product_id: int
    name: str | None
    
value = {
    "product_id" : "50"
}

try:
    p1 = Product(**value)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of error: 1
Fields : ('name',)  , Type : missing ,  Message: Field required


what happens if value is None

In [15]:
value = {
    "product_id" : "50",
    "name": None
}

try:
    p1 = Product(**value)
    print(p1)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

product_id=50 name=None


> **Case C — Optional field and non-nullable**

 what happens when value is not provided in dict 

In [22]:
class Product(BaseModel):
    product_id: int
    name: str = "Ram"

value = {
    "product_id" : "50"
}

try:
    p1 = Product(**value)
    print(p1)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

product_id=50 name='Ram'


what happens if value is None

In [2]:
from pydantic import BaseModel, ValidationError
class Product(BaseModel):
    product_id: int
    name: str = "Ram"

value = {
    "product_id" : "50",
    "name": None
}

try:
    p1 = Product(**value)
    print(p1)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of error: 1
Fields : ('name',)  , Type : string_type ,  Message: Input should be a valid string


> **Case D — Optional field and nullable**

In [26]:
class Product(BaseModel):
    product_id: int
    name: str | None = None

value = {
    "product_id" : "50"
}

try:
    p1 = Product(**value)
    print(p1)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

product_id=50 name=None


what happen if value is None

In [28]:
class Product(BaseModel):
    product_id: int
    name: str | None = None

value = {
    "product_id" : "50",
    "name": None
}

try:
    p1 = Product(**value)
    print(p1)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

product_id=50 name=None


> **pydantic does't validate for default value by itself.**

In [ ]:
class Product(BaseModel):
    product_id: int = "abc"
try:
    p1 = Product()
    print(p1)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

product_id='abc'


> #### **Field()**

for difference instances get same datetime, so it should be different datetime

In [10]:
from datetime import datetime, UTC
import time 

class BadExample(BaseModel):
    created_dt: datetime = datetime.now(UTC)
    
b1 = BadExample()
time.sleep(2)
b2 = BadExample()
print(b1)
print(b1.created_dt)
print(b2.created_dt)
print(b1.created_dt == b2.created_dt)

created_dt=datetime.datetime(2026, 11, 27, 2, 5, 17, 158320, tzinfo=datetime.timezone.utc)
2026-11-27 02:05:17.158320+00:00
2026-11-27 02:05:17.158320+00:00
True


`In Python, **uuid** is a built-in library used to generate Universally Unique Identifiers (UUIDs).`
- **UUID**: Used to create or work with unique identifiers.
- **uuid4()**: Generates a random unique ID.

**Why use it?**
- To generate unique IDs for users, records, or tokens.
- To avoid duplicate identifiers.
- To uniquely identify database records.

**In simple words: uuid4() generates a random unique ID, while UUID helps work with UUID values.**

**`default_factory means generate a fresh default value every time a new object is created.`**
- This is useful when working with APIs, request tracking, and database records.

In [ ]:
from uuid import UUID, uuid4
from pydantic import Field
import time
class GoodExample(BaseModel):
    # UUID - Data type.
    request_id: UUID = Field(default_factory=uuid4) #Automatically generates a new unique ID whenever an object is created.
    created_at: datetime = Field(default_factory=lambda: datetime.now(UTC)) #Automatically generates the current time whenever an object is created.
    
g1 = GoodExample()
time.sleep(2)
g2 = GoodExample()
print(g1)
print(g1.created_at)
print(g2.created_at)
print(g1.created_at == g2.created_at)

request_id=UUID('676f9602-fedd-494b-b039-9f5ced051cff') created_at=datetime.datetime(2026, 11, 27, 2, 5, 4, 408057, tzinfo=datetime.timezone.utc)
2026-11-27 02:05:04.408057+00:00
2026-11-27 02:05:06.408483+00:00
False


> **Production Example:**

**Pydantic models use dot notation to access fields.**

In [12]:
from datetime import datetime, UTC
from uuid import UUID, uuid4
from pydantic import BaseModel, Field

class LLMChatRequest(BaseModel):
    # Required
    prompt: str
    
    # Optional with sensible production defaults
    model: str = "gpt-4o-mini"
    temperature: float = 0.7
    max_token: int | None = None        # None = let provider use model default
    system_prompt: str | None = None    # Optional system instruction
    stop_sequences: list[str] = Field(default_factory=list)
    
    # Auto-generated metadata per request
    trace_id: UUID = Field(default_factory=uuid4)
    created_at: datetime = Field(default_factory=lambda: datetime.now(UTC))
    
req = LLMChatRequest(prompt="Explain RAG in 2 sentences.")
print(req)

req.stop_sequences.append("hello stop")
print(req)

prompt='Explain RAG in 2 sentences.' model='gpt-4o-mini' temperature=0.7 max_token=None system_prompt=None stop_sequences=[] trace_id=UUID('d09be571-4230-4329-beb7-2cc7d5a085be') created_at=datetime.datetime(2026, 11, 27, 2, 7, 53, 88999, tzinfo=datetime.timezone.utc)
prompt='Explain RAG in 2 sentences.' model='gpt-4o-mini' temperature=0.7 max_token=None system_prompt=None stop_sequences=['hello stop'] trace_id=UUID('d09be571-4230-4329-beb7-2cc7d5a085be') created_at=datetime.datetime(2026, 11, 27, 2, 7, 53, 88999, tzinfo=datetime.timezone.utc)


In [43]:
class Prompt(BaseModel):
    text: str = Field(description="user text value")
    user_id: str = Field(...,description="caller user id")
    value: str | None = Field(default=None,description="random value")
    
print(Prompt(text="abc",user_id="pqr",value=None))

text='abc' user_id='pqr' value=None


**Validate_default parameter in Field():**

How to validate default value 


In [46]:
class ValidateVal(BaseModel):
    product_id: int = Field(default="xyz",validate_default=True)
    
try:
    ValidateVal()
except ValidationError as e:
    print(e.errors())
    print(e.errors()[0])
    print(e.errors()[0]['msg'])

[{'type': 'int_parsing', 'loc': ('product_id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'xyz', 'url': 'https://errors.pydantic.dev/2.14/v/int_parsing'}]
{'type': 'int_parsing', 'loc': ('product_id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'xyz', 'url': 'https://errors.pydantic.dev/2.14/v/int_parsing'}
Input should be a valid integer, unable to parse string as an integer


Dynamic data assign

In [47]:
class ModelDate(BaseModel):
    model_name: str
    version: int = 1
    attribute_val: str = Field(default_factory=lambda data:f"{data['model_name']}:{data['version']}")
    
a = ModelDate(model_name="gpt")
print(a)

model_name='gpt' version=1 attribute_val='gpt:1'


**creates an ML Experiment Tracker using Pydantic**

In [20]:
"""
Create an ExperimentRun(BaseModel) for an ML training tracker with:

experiment_name: str (required, non-nullable)
parent_run_id: str | None (required key, can be None)
learning_rate: float = 0.001 (optional, non-nullable)
run_id: UUID (auto-generated fresh UUID per instance via default_factory)
metrics: dict[str, float] (defaults to a fresh empty dict {} via default_factory)

Test creating 2 separate runs and verify run1.run_id != run2.run_id!
"""

from uuid import UUID, uuid4
from pydantic import BaseModel, Field

class ExperimentRun(BaseModel):
    experiment_name: str
    parent_run_id: str | None
    learning_rate: float = 0.001
    run_id: UUID = Field(default_factory=uuid4)
    metrics: dict[str, float] = Field(default_factory=dict)
    
run1 = ExperimentRun(experiment_name="Test Experiment", parent_run_id=None)
run2 = ExperimentRun(experiment_name="Test Experiment", parent_run_id="RUN-001")
print(run1)
print(run2)
print(run1.run_id != run2.run_id)


experiment_name='Test Experiment' parent_run_id=None learning_rate=0.001 run_id=UUID('cfdfe936-8e28-4f4c-983d-6f28086bd718') metrics={}
experiment_name='Test Experiment' parent_run_id='RUN-001' learning_rate=0.001 run_id=UUID('82357f93-f1c4-462a-a1b8-16a074d58461') metrics={}
True


In [ ]:

from pydantic import BaseModel, Field
from uuid import UUID, uuid4

class ExperimentRun(BaseModel):
    experiment_name: str
    parent_run_id: str | None
    learning_rate: float = 0.001
    run_id: UUID = Field(default_factory=uuid4)
    metrics: dict[str, float] = Field(default_factory=dict)


# Create two separate experiment runs
run1 = ExperimentRun(experiment_name="Model Training",parent_run_id=None)
run2 = ExperimentRun(experiment_name="Model Testing",parent_run_id="RUN-001")

# Print both runs
print("Run 1:", run1)
print("Run 2:", run2)

# Verify UUIDs are different
print("\nRun 1 ID:", run1.run_id)
print("Run 2 ID:", run2.run_id)
print("Are UUIDs different?", run1.run_id != run2.run_id)

# Test independent dictionaries
run1.metrics["accuracy"] = 0.95

print("\nRun 1 Metrics:", run1.metrics)
print("Run 2 Metrics:", run2.metrics)


Run 1: experiment_name='Model Training' parent_run_id=None learning_rate=0.001 run_id=UUID('73c3ae73-4277-4c91-8a74-a4966d575a84') metrics={}
Run 2: experiment_name='Model Testing' parent_run_id='RUN-001' learning_rate=0.001 run_id=UUID('956b6833-1331-42ad-92db-f78a01d6536e') metrics={}

Run 1 ID: 73c3ae73-4277-4c91-8a74-a4966d575a84
Run 2 ID: 956b6833-1331-42ad-92db-f78a01d6536e
Are UUIDs different? True

Run 1 Metrics: {'accuracy': 0.95}
Run 2 Metrics: {}


> #### **field_validator**